# 03 — Financial Analysis and NLP

This notebook demonstrates the analytical layer that transforms parsed
XBRL data and filing text into quantitative features:

1. **Financial metrics** — ratios, growth rates, FCF, inflection points
2. **TTM computation** — trailing twelve months from quarterly data
3. **Narrative drift** — TF-IDF cosine similarity between consecutive filings
4. **Keyword scoring** — normalised frequency across 9 tracked themes
5. **Shift snippets** — examples of significant language changes

Modules: `src/financial_metrics.py`, `src/nlp_features.py`

> **Note:** The authoritative reproducibility path is `python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01 --output-format both`. This notebook is an explanatory wrapper that inspects the same pipeline outputs.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from pathlib import Path

from src.config import get_default_config, KEYWORD_DICTIONARIES
from src.financial_metrics import FinancialMetricsCalculator
from src.nlp_features import NLPFeatureExtractor

In [ ]:
config = get_default_config()

## 1. Financial Metrics

`FinancialMetricsCalculator.compute_all_metrics()` takes parsed XBRL
data and produces per-period ratios:

- **Margins**: gross, operating, net
- **Returns**: ROE, ROA
- **Cash flow**: FCF, FCF margin
- **Leverage**: debt-to-equity, current ratio
- **Expense**: R&D % revenue, SG&A % revenue
- **Growth**: revenue YoY, operating income YoY
- **Per-share**: diluted EPS

Every row carries `source_available_date` for point-in-time controls.

In [ ]:
metrics_path = config.processed_dir / "nvda_metrics.csv"

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(f"Metrics rows: {len(metrics)}")
    print(f"Unique metrics: {metrics['metric_name'].nunique()}")
    print(f"Periods: {sorted(metrics['fiscal_period'].unique())}")
    metrics.head(15)
else:
    print("No metrics CSV yet — run the pipeline first.")
    metrics = None

### Margin Trends

A quick look at how Nvidia's margins have evolved over time.

In [ ]:
if metrics is not None:
    margin_names = ["gross_margin", "operating_margin", "net_margin"]
    margins = metrics[metrics["metric_name"].isin(margin_names)]
    pivot = margins.pivot_table(
        index="fiscal_period", columns="metric_name",
        values="metric_value", aggfunc="first",
    ).sort_index()
    print("Margin trends (most recent periods):")
    pivot.tail(8)

### Inflection Points

`flag_inflection_points()` marks metrics where the YoY change exceeds
2 standard deviations from the historical mean — a signal of potential
business inflection.

In [ ]:
if metrics is not None:
    calc = FinancialMetricsCalculator(config)
    flagged = calc.flag_inflection_points(metrics)
    inflections = flagged[flagged["inflection_flag"] == True]
    print(f"Inflection points flagged: {len(inflections)}")
    if not inflections.empty:
        inflections[["fiscal_period", "metric_name", "metric_value"]].head(10)

## 2. Narrative Drift (NLP)

The NLP layer measures how filing language changes over time:

- **TF-IDF cosine similarity** between consecutive filings for Risk Factors
  and MD&A — a drop signals new disclosures or changed risk emphasis
- **Keyword scoring** across 9 themes — tracks the relative prominence
  of topics like AI, export controls, competition, etc.

### Tracked Themes

In [ ]:
extractor = NLPFeatureExtractor(config)

print("9 tracked themes:")
for i, theme in enumerate(extractor.TRACKED_THEMES, 1):
    keywords = KEYWORD_DICTIONARIES[theme][:4]
    print(f"  {i}. {theme}: {', '.join(keywords)}, ...")

### Keyword Scoring Example

`compute_keyword_scores()` returns normalised frequency (count / total words)
for each theme. Here's a quick demo on sample text.

In [ ]:
sample_text = (
    "NVIDIA's data center revenue grew significantly driven by "
    "artificial intelligence and accelerated computing demand. "
    "Export controls on China remain a risk factor. Competition "
    "from AMD and custom silicon alternatives like TPU is increasing. "
    "Gaming revenue showed cyclical recovery."
)

scores = extractor.compute_keyword_scores(sample_text)
print("Keyword scores (sample text):")
for theme, score in sorted(scores.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 5000)
    print(f"  {theme:35s} {score:.6f}  {bar}")

### Full NLP Feature Matrix

The pre-computed NLP features are saved to `data/processed/nvda_nlp_features.csv`.

In [ ]:
nlp_path = config.processed_dir / "nvda_nlp_features.csv"

if nlp_path.exists():
    nlp = pd.read_csv(nlp_path)
    print(f"NLP feature rows: {len(nlp)}")
    print(f"Feature types: {nlp['feature_type'].unique().tolist()}")
    print(f"Sections: {nlp['section'].unique().tolist()}")
    
    # Show TF-IDF similarity scores
    tfidf = nlp[nlp["feature_type"] == "tfidf_similarity"]
    if not tfidf.empty:
        print("\nTF-IDF similarity (lower = more change):")
        tfidf[["filing_date", "section", "value"]].head(10)
else:
    print("No NLP features yet — run the pipeline first.")

---

**Next:** [04_ml_model.ipynb](04_ml_model.ipynb) —
feature/target matrix, model training, walk-forward validation, and audit.